## Cell 1 — Install dependencies

In [1]:
!pip install psycopg2-binary pandas numpy scikit-learn imbalanced-learn \
             xgboost lightgbm matplotlib seaborn joblib ta python-dotenv vaderSentiment sqlalchemy

## Cell 2 — Imports & DB connection

In [2]:
import os
import re
import json
import numpy as np
import pandas as pd
import psycopg2
from sqlalchemy import create_engine, text

# ── DB credentials from environment variables ──────────────────────────────
DB_USER     = os.environ["DB_USER"]
DB_PASSWORD = os.environ["DB_PASSWORD"]
DB_HOST     = os.environ["DB_HOST"]
DB_DATABASE = os.environ["DB_DATABASE"]
DB_PORT     = 5432

conn_str = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_DATABASE}"
)
engine = create_engine(conn_str)

print("✅ Connection string built. Attempting test query…")
with engine.connect() as c:
    result = c.execute(text("SELECT 1")).fetchone()
    print("✅ DB connected:", result)


✅ Connection string built. Attempting test query…
✅ DB connected: (1,)


## Cell 3 — Load daily price data for Bitcoin, Ethereum, and Dogecoin

In [3]:
price_query = text("""
SELECT "Currency_Name",
       "Date",
       "Price",
       "Open",
       "High",
       "Low",
       "Vol.",
       "Change %%" AS change_pct
FROM api.top_cryptocurrencies_historical_prices
WHERE "Currency_Name" IN ('Bitcoin', 'Ethereum', 'Dogecoin')
ORDER BY "Currency_Name", "Date"
""")

with engine.connect() as conn:
    raw_price = pd.read_sql(price_query, conn)

print(raw_price.shape)
print(raw_price.dtypes)
raw_price.head(10)


(7594, 8)
Currency_Name     object
Date              object
Price            float64
Open             float64
High             float64
Low              float64
Vol.             float64
change_pct       float64
dtype: object


,Currency_Name,Date,Price,Open,High,Low,Vol.,change_pct
0,Bitcoin,2010-07-18,0.1,0.0,0.1,0.1,80.0,0.0
1,Bitcoin,2010-07-19,0.1,0.1,0.1,0.1,570.0,0.0
2,Bitcoin,2010-07-20,0.1,0.1,0.1,0.1,260.0,0.0
3,Bitcoin,2010-07-21,0.1,0.1,0.1,0.1,580.0,0.0
4,Bitcoin,2010-07-22,0.1,0.1,0.1,0.1,2160.0,0.0
5,Bitcoin,2010-07-23,0.1,0.1,0.1,0.1,2400.0,0.0
6,Bitcoin,2010-07-24,0.1,0.1,0.1,0.1,500.0,0.0
7,Bitcoin,2010-07-25,0.1,0.1,0.1,0.1,1550.0,0.0
8,Bitcoin,2010-07-26,0.1,0.1,0.1,0.1,880.0,0.0
9,Bitcoin,2010-07-27,0.1,0.1,0.1,0.1,3370.0,0.0


## Cell 4 — Parse and clean the price columns

In [4]:
def clean_price_col(series):
    """Remove commas, K/M/B suffixes, % signs — return float."""
    s = series.astype(str).str.replace(",", "", regex=False)
    s = s.str.replace("%", "", regex=False).str.strip()
    multiplier = s.str.endswith("K").map({True: 1e3, False: 1})
    multiplier *= s.str.endswith("M").map({True: 1e6, False: 1})
    multiplier *= s.str.endswith("B").map({True: 1e9, False: 1})
    s = s.str.rstrip("KMBkmb")
    return pd.to_numeric(s, errors="coerce") * multiplier

raw_price["Date"] = pd.to_datetime(raw_price["Date"], infer_datetime_format=True)
raw_price["close"]  = clean_price_col(raw_price["Price"])
raw_price["open"]   = clean_price_col(raw_price["Open"])
raw_price["high"]   = clean_price_col(raw_price["High"])
raw_price["low"]    = clean_price_col(raw_price["Low"])
raw_price["volume"] = clean_price_col(raw_price["Vol."])
raw_price["pct_change_raw"] = clean_price_col(raw_price["change_pct"])

price = (
    raw_price[["Currency_Name", "Date", "open", "high", "low", "close", "volume", "pct_change_raw"]]
    .rename(columns={"Currency_Name": "coin", "Date": "date"})
    .sort_values(["coin", "date"])
    .drop_duplicates(subset=["coin", "date"], keep="last")
    .reset_index(drop=True)
)

print(price.dtypes)
price.head()


coin                      object
date              datetime64[ns]
open                     float64
high                     float64
low                      float64
close                    float64
volume                   float64
pct_change_raw           float64
dtype: object


,coin,date,open,high,low,close,volume,pct_change_raw
0,Bitcoin,2010-07-18,0.0,0.1,0.1,0.1,80.0,0.0
1,Bitcoin,2010-07-19,0.1,0.1,0.1,0.1,570.0,0.0
2,Bitcoin,2010-07-20,0.1,0.1,0.1,0.1,260.0,0.0
3,Bitcoin,2010-07-21,0.1,0.1,0.1,0.1,580.0,0.0
4,Bitcoin,2010-07-22,0.1,0.1,0.1,0.1,2160.0,0.0


## Cell 5 — Check date coverage per coin

In [5]:
for coin, g in price.groupby("coin"):
    print(f"{coin:12s}  rows={len(g):,}  "
          f"from={g['date'].min().date()}  to={g['date'].max().date()}")


Bitcoin       rows=4,056  from=2010-07-18  to=2021-08-24
Dogecoin      rows=1,544  from=2017-06-03  to=2021-08-24
Ethereum      rows=1,994  from=2016-03-10  to=2021-08-24


## Cell 6 — Locate the large Bitcoin Twitter CSV in the workspace

In [6]:
import os
from pathlib import Path
import csv

print("Current working directory:", os.getcwd())

# Search for the large BTC Twitter CSV in common workspace locations
search_roots = [
    Path("."),
    Path("./downloads"),
    Path("/mnt/data"),
    Path("/mnt/data/downloads"),
    Path.home(),
]

target_name = "engtweetsbtc_vader_sentiment.csv"
matches = []

for root in search_roots:
    try:
        if root.exists():
            matches.extend(root.rglob(target_name))
    except Exception:
        pass

# Remove duplicates while preserving order
seen = set()
unique_matches = []
for m in matches:
    try:
        resolved = str(m.resolve())
    except Exception:
        resolved = str(m)
    if resolved not in seen:
        seen.add(resolved)
        unique_matches.append(m)

print("Found matches:")
for m in unique_matches[:20]:
    print(" -", m)

if not unique_matches:
    raise FileNotFoundError(
        "Could not find engtweetsbtc_vader_sentiment.csv anywhere reachable from this notebook. "
        "Move/copy it into the notebook workspace (for example ./downloads/) or upload it directly."
    )

twitter_csv_path = str(unique_matches[0])
print("\nUsing Twitter CSV:", twitter_csv_path)

# Detect separator robustly
with open(twitter_csv_path, "r", encoding="utf-8", errors="ignore") as f:
    sample = f.read(5000)

try:
    twitter_sep = csv.Sniffer().sniff(sample, delimiters=",;|\t").delimiter
except Exception:
    twitter_sep = ";"

twitter_preview = pd.read_csv(twitter_csv_path, sep=twitter_sep, nrows=5, low_memory=False)

print("Detected separator:", repr(twitter_sep))
print("Preview shape:", twitter_preview.shape)
print("Columns:", twitter_preview.columns.tolist())
twitter_preview.head()

Current working directory: /home/jovyan/updated_crypto_deploy_pack
Found matches:
 - /home/jovyan/downloads/old_btc_twitter/engtweetsbtc_vader_sentiment.csv

Using Twitter CSV: /home/jovyan/downloads/old_btc_twitter/engtweetsbtc_vader_sentiment.csv
Detected separator: ';'
Preview shape: (5, 22)
Columns: ['origen', 'date', 'username', 'user_fullname', 'user_description', 'user_created', 'user_verified', 'location', 'n_followers', 'n_following', 'user_favourites', 'n_replies', 'n_likes', 'n_retweets', 'hashtags', 'url', 'source', 'is_retweet', 'text', 'tweet_language', 'vader_compound', 'vader_sentiment']


,origen,date,username,user_fullname,user_description,user_created,user_verified,location,n_followers,n_following,...,n_likes,n_retweets,hashtags,url,source,is_retweet,text,tweet_language,vader_compound,vader_sentiment
0,df1,2019-05-27 11:49:18+00:00,bitcointe,Bitcointe,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,NaN,NaN,NaN,NaN,Cardano: Digitize Currencies; EOS 6500% ROI; ...,en,-0.1027,NEGATIVE
1,df1,2019-05-27 11:49:06+00:00,3eyedbran,Bran - 3 Eyed Raven,NaN,NaN,NaN,NaN,NaN,NaN,...,2,1,NaN,NaN,NaN,NaN,Another Test tweet that was not caught in the ...,en,0.0000,NEUTRAL
2,df1,2019-05-27 11:49:22+00:00,DetroitCrypto,J. Scardina,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,NaN,NaN,NaN,NaN,Current Crypto Prices! \n\nBTC: $8721.99 USD\n...,en,0.0000,NEUTRAL
3,df1,2019-05-27 11:49:25+00:00,evilrobotted,evilrobotted,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,NaN,NaN,NaN,NaN,We have been building on the real #bitcoin SV....,en,-0.4767,NEGATIVE
4,df1,2019-05-22 12:42:16+00:00,Cybintelligence,CybIntelligence,NaN,NaN,NaN,NaN,NaN,NaN,...,2,7,NaN,NaN,NaN,NaN,BestMixer has been seized by the Dutch Police ...,en,0.0000,NEUTRAL


## Cell 7 — Load the large Bitcoin Twitter CSV

This notebook now prefers the large CSV-based Bitcoin Twitter dataset rather than the tiny
`api.stock_market_tweets` table.

Important:
- The CSV is already Bitcoin-focused.
- It already contains VADER sentiment columns.
- Later notebooks should use:
  - **Bitcoin** → price + BTC Twitter features
  - **Ethereum** → price only
  - **Dogecoin** → price only

In [7]:
# Load the full large Bitcoin Twitter CSV
raw_twitter = pd.read_csv(twitter_csv_path, sep=twitter_sep, low_memory=False)

print("raw_twitter shape:", raw_twitter.shape)
print(raw_twitter.dtypes.head(10))

required_cols = [
    "date",
    "text",
    "tweet_language",
    "is_retweet",
    "vader_compound",
    "vader_sentiment",
]

missing = [c for c in required_cols if c not in raw_twitter.columns]
if missing:
    raise ValueError(f"Missing expected Twitter columns: {missing}")

raw_twitter = raw_twitter[required_cols].copy()

# Standardise types
raw_twitter["date"] = pd.to_datetime(raw_twitter["date"], errors="coerce", utc=True)
raw_twitter["text"] = raw_twitter["text"].astype(str)
raw_twitter["tweet_language"] = raw_twitter["tweet_language"].astype(str).str.lower().str.strip()
raw_twitter["vader_compound"] = pd.to_numeric(raw_twitter["vader_compound"], errors="coerce")
raw_twitter["is_retweet"] = raw_twitter["is_retweet"].astype(str).str.lower().str.strip()

# Keep valid English tweet rows with usable sentiment
twitter_clean = raw_twitter[
    raw_twitter["date"].notna() &
    raw_twitter["text"].str.strip().ne("") &
    raw_twitter["vader_compound"].notna() &
    raw_twitter["tweet_language"].eq("en")
].copy()

print("twitter_clean shape:", twitter_clean.shape)
print("twitter_clean date range:",
      twitter_clean["date"].min(),
      "to",
      twitter_clean["date"].max())
twitter_clean.head()

raw_twitter shape: (26344493, 22)
origen               object
date                 object
username             object
user_fullname        object
user_description     object
user_created         object
user_verified        object
location             object
n_followers         float64
n_following         float64
dtype: object
twitter_clean shape: (26344492, 6)
twitter_clean date range: 2013-01-01 05:12:29+00:00 to 2023-05-28 00:00:00+00:00


,date,text,tweet_language,is_retweet,vader_compound,vader_sentiment
0,2019-05-27 11:49:18+00:00,Cardano: Digitize Currencies; EOS 6500% ROI; ...,en,nan,-0.1027,NEGATIVE
1,2019-05-27 11:49:06+00:00,Another Test tweet that was not caught in the ...,en,nan,0.0000,NEUTRAL
2,2019-05-27 11:49:22+00:00,Current Crypto Prices! \n\nBTC: $8721.99 USD\n...,en,nan,0.0000,NEUTRAL
3,2019-05-27 11:49:25+00:00,We have been building on the real #bitcoin SV....,en,nan,-0.4767,NEGATIVE
4,2019-05-22 12:42:16+00:00,BestMixer has been seized by the Dutch Police ...,en,nan,0.0000,NEUTRAL


## Cell 8 — Use the CSV’s existing Bitcoin sentiment values

In [8]:
# This CSV is already Bitcoin-focused and already has VADER sentiment
# So we do NOT need keyword detection or re-scoring here

twitter_btc = twitter_clean.copy()
twitter_btc["coin"] = "Bitcoin"
twitter_btc["sentiment_score"] = twitter_btc["vader_compound"]

# Convert to naive daily date for merging with daily price data
twitter_btc["date"] = twitter_btc["date"].dt.tz_localize(None).dt.floor("D")

# Keep only rows overlapping the Bitcoin price history
btc_min = price.loc[price["coin"] == "Bitcoin", "date"].min()
btc_max = price.loc[price["coin"] == "Bitcoin", "date"].max()

twitter_btc = twitter_btc[
    (twitter_btc["date"] >= btc_min) &
    (twitter_btc["date"] <= btc_max)
].copy()

# Optional: drop exact duplicate tweet records on same date/text/sentiment
twitter_btc = twitter_btc.drop_duplicates(subset=["date", "text", "sentiment_score"])

print("Bitcoin tweet rows after cleaning:", len(twitter_btc))
print("Bitcoin tweet date range:",
      twitter_btc["date"].min(),
      "to",
      twitter_btc["date"].max())
print("Unique Bitcoin tweet days:", twitter_btc["date"].nunique())

twitter_btc[["date", "text", "coin", "sentiment_score", "vader_sentiment"]].head()

Bitcoin tweet rows after cleaning: 14611637
Bitcoin tweet date range: 2013-01-01 00:00:00 to 2021-08-24 00:00:00
Unique Bitcoin tweet days: 2754


,date,text,coin,sentiment_score,vader_sentiment
0,2019-05-27,Cardano: Digitize Currencies; EOS 6500% ROI; ...,Bitcoin,-0.1027,NEGATIVE
1,2019-05-27,Another Test tweet that was not caught in the ...,Bitcoin,0.0000,NEUTRAL
2,2019-05-27,Current Crypto Prices! \n\nBTC: $8721.99 USD\n...,Bitcoin,0.0000,NEUTRAL
3,2019-05-27,We have been building on the real #bitcoin SV....,Bitcoin,-0.4767,NEGATIVE
4,2019-05-22,BestMixer has been seized by the Dutch Police ...,Bitcoin,0.0000,NEUTRAL


## Cell 9 — Aggregate Bitcoin sentiment to daily features

In [9]:
btc_twitter_daily = (
    twitter_btc
    .groupby(["coin", "date"], as_index=False)
    .agg(
        btc_twitter_sentiment_mean=("sentiment_score", "mean"),
        btc_twitter_sentiment_std=("sentiment_score", "std"),
        btc_twitter_tweet_count=("sentiment_score", "size"),
    )
)

btc_twitter_daily["btc_twitter_sentiment_std"] = (
    btc_twitter_daily["btc_twitter_sentiment_std"].fillna(0)
)

print("btc_twitter_daily shape:", btc_twitter_daily.shape)
print("btc_twitter_daily date range:",
      btc_twitter_daily["date"].min(),
      "to",
      btc_twitter_daily["date"].max())

nonzero_days = (btc_twitter_daily["btc_twitter_tweet_count"] > 0).sum()
print("Bitcoin daily Twitter rows with tweets:", nonzero_days)

btc_twitter_daily.head()

btc_twitter_daily shape: (2754, 5)
btc_twitter_daily date range: 2013-01-01 00:00:00 to 2021-08-24 00:00:00
Bitcoin daily Twitter rows with tweets: 2754


,coin,date,btc_twitter_sentiment_mean,btc_twitter_sentiment_std,btc_twitter_tweet_count
0,Bitcoin,2013-01-01,-0.005627,0.166366,11
1,Bitcoin,2013-01-02,0.020000,0.082462,17
2,Bitcoin,2013-01-03,0.102525,0.189335,24
3,Bitcoin,2013-01-04,-0.006246,0.208878,28
4,Bitcoin,2013-01-05,0.043478,0.232035,27


## Cell 10 — Build separate modelling datasets by coin

In [10]:
bitcoin_price  = price[price["coin"] == "Bitcoin"].copy()
ethereum_price = price[price["coin"] == "Ethereum"].copy()
dogecoin_price = price[price["coin"] == "Dogecoin"].copy()

# Bitcoin gets Bitcoin-only Twitter features
bitcoin_model_data = bitcoin_price.merge(
    btc_twitter_daily,
    on=["coin", "date"],
    how="left"
)

bitcoin_model_data["btc_twitter_sentiment_mean"] = bitcoin_model_data["btc_twitter_sentiment_mean"].fillna(0)
bitcoin_model_data["btc_twitter_sentiment_std"] = bitcoin_model_data["btc_twitter_sentiment_std"].fillna(0)
bitcoin_model_data["btc_twitter_tweet_count"] = bitcoin_model_data["btc_twitter_tweet_count"].fillna(0)

# Ethereum and Dogecoin stay price-only
ethereum_model_data = ethereum_price.copy()
dogecoin_model_data = dogecoin_price.copy()

print("Bitcoin model shape: ", bitcoin_model_data.shape)
print("Ethereum model shape:", ethereum_model_data.shape)
print("Dogecoin model shape:", dogecoin_model_data.shape)
bitcoin_model_data.head()


Bitcoin model shape:  (4056, 11)
Ethereum model shape: (1994, 8)
Dogecoin model shape: (1544, 8)


,coin,date,open,high,low,close,volume,pct_change_raw,btc_twitter_sentiment_mean,btc_twitter_sentiment_std,btc_twitter_tweet_count
0,Bitcoin,2010-07-18,0.0,0.1,0.1,0.1,80.0,0.0,0.0,0.0,0.0
1,Bitcoin,2010-07-19,0.1,0.1,0.1,0.1,570.0,0.0,0.0,0.0,0.0
2,Bitcoin,2010-07-20,0.1,0.1,0.1,0.1,260.0,0.0,0.0,0.0,0.0
3,Bitcoin,2010-07-21,0.1,0.1,0.1,0.1,580.0,0.0,0.0,0.0,0.0
4,Bitcoin,2010-07-22,0.1,0.1,0.1,0.1,2160.0,0.0,0.0,0.0,0.0


## Cell 11 — Quality checks: missing values

In [11]:
print("=== PRICE missing values ===")
print(price.isnull().sum())

print("\n=== BITCOIN model missing values ===")
print(bitcoin_model_data.isnull().sum())

print("\n=== ETHEREUM model missing values ===")
print(ethereum_model_data.isnull().sum())

print("\n=== DOGECOIN model missing values ===")
print(dogecoin_model_data.isnull().sum())


=== PRICE missing values ===
coin              0
date              0
open              0
high              0
low               0
close             0
volume            0
pct_change_raw    0
dtype: int64

=== BITCOIN model missing values ===
coin                          0
date                          0
open                          0
high                          0
low                           0
close                         0
volume                        0
pct_change_raw                0
btc_twitter_sentiment_mean    0
btc_twitter_sentiment_std     0
btc_twitter_tweet_count       0
dtype: int64

=== ETHEREUM model missing values ===
coin              0
date              0
open              0
high              0
low               0
close             0
volume            0
pct_change_raw    0
dtype: int64

=== DOGECOIN model missing values ===
coin              0
date              0
open              0
high              0
low               0
close             0
volume            0
pct_

## Cell 12 — Quality checks: duplicates

In [12]:
price_dupes = price.duplicated(subset=["coin", "date"]).sum()
btc_twitter_dupes = btc_twitter_daily.duplicated(subset=["coin", "date"]).sum()
btc_model_dupes = bitcoin_model_data.duplicated(subset=["coin", "date"]).sum()
eth_model_dupes = ethereum_model_data.duplicated(subset=["coin", "date"]).sum()
doge_model_dupes = dogecoin_model_data.duplicated(subset=["coin", "date"]).sum()

print(f"Price duplicate (coin, date) rows:        {price_dupes}")
print(f"BTC Twitter duplicate (coin, date) rows:  {btc_twitter_dupes}")
print(f"Bitcoin model duplicate rows:             {btc_model_dupes}")
print(f"Ethereum model duplicate rows:            {eth_model_dupes}")
print(f"Dogecoin model duplicate rows:            {doge_model_dupes}")


Price duplicate (coin, date) rows:        0
BTC Twitter duplicate (coin, date) rows:  0
Bitcoin model duplicate rows:             0
Ethereum model duplicate rows:            0
Dogecoin model duplicate rows:            0


## Cell 13 — Bitcoin Twitter coverage check

In [13]:
print("BTC Twitter daily rows:", len(btc_twitter_daily))
if len(btc_twitter_daily) > 0:
    print("BTC Twitter date range:",
          btc_twitter_daily["date"].min().date(),
          "to",
          btc_twitter_daily["date"].max().date())

nonzero_days = (bitcoin_model_data["btc_twitter_tweet_count"] > 0).sum()
print("Bitcoin price rows with non-zero Twitter coverage:", nonzero_days)
print("Bitcoin price rows total:", len(bitcoin_model_data))
print("Coverage ratio:", f"{nonzero_days / len(bitcoin_model_data):.2%}" if len(bitcoin_model_data) else "n/a")


BTC Twitter daily rows: 2754
BTC Twitter date range: 2013-01-01 to 2021-08-24
Bitcoin price rows with non-zero Twitter coverage: 2754
Bitcoin price rows total: 4056
Coverage ratio: 67.90%


## Cell 14 — Outlier check for daily close price

In [14]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (coin, g) in zip(axes, price.groupby("coin")):
    ax.boxplot(g["close"].dropna())
    ax.set_title(f"{coin} Close Price")
    ax.set_ylabel("Price (USD)")

plt.suptitle("Outlier Check — Daily Close Price")
plt.tight_layout()
plt.savefig("quality_outlier_boxplot.png", dpi=120)
plt.show()
print("✅ Saved quality_outlier_boxplot.png")


✅ Saved quality_outlier_boxplot.png


## Cell 15 — Preview class balance for each coin

In [15]:
def add_target(df):
    out = df.sort_values("date").copy()
    out["target"] = (out["close"].shift(-1) > out["close"]).astype(int)
    return out

bitcoin_preview = add_target(bitcoin_model_data)
ethereum_preview = add_target(ethereum_model_data)
dogecoin_preview = add_target(dogecoin_model_data)

for name, df in [
    ("Bitcoin", bitcoin_preview),
    ("Ethereum", ethereum_preview),
    ("Dogecoin", dogecoin_preview),
]:
    counts = df["target"].value_counts()
    ratio = counts.get(1, 0) / len(df.dropna(subset=["target"]))
    print(f"{name:12s}  Up={counts.get(1,0)}  Down={counts.get(0,0)}  "
          f"Up%={ratio:.1%}  ← {'balanced' if 0.4 < ratio < 0.6 else 'IMBALANCED'}")


Bitcoin       Up=1972  Down=2084  Up%=48.6%  ← balanced
Ethereum      Up=1015  Down=979  Up%=50.9%  ← balanced
Dogecoin      Up=748  Down=796  Up%=48.4%  ← balanced


## Cell 16 — Leakage sanity check

In [16]:
btc = bitcoin_preview.sort_values("date").head(5)
print(btc[["date", "open", "close", "target"]].to_string(index=False))
print("\n✅ Features use today-or-earlier data; target is next-day direction.")


      date  open  close  target
2010-07-18   0.0    0.1       0
2010-07-19   0.1    0.1       0
2010-07-20   0.1    0.1       0
2010-07-21   0.1    0.1       0
2010-07-22   0.1    0.1       0

✅ Features use today-or-earlier data; target is next-day direction.


## Cell 17 — Define feature sets for later notebooks

In [17]:
feature_sets = {
    "Bitcoin": [
        "open", "high", "low", "close", "volume", "pct_change_raw",
        "btc_twitter_sentiment_mean", "btc_twitter_sentiment_std", "btc_twitter_tweet_count"
    ],
    "Ethereum": [
        "open", "high", "low", "close", "volume", "pct_change_raw"
    ],
    "Dogecoin": [
        "open", "high", "low", "close", "volume", "pct_change_raw"
    ]
}

print(json.dumps(feature_sets, indent=2))


{
  "Bitcoin": [
    "open",
    "high",
    "low",
    "close",
    "volume",
    "pct_change_raw",
    "btc_twitter_sentiment_mean",
    "btc_twitter_sentiment_std",
    "btc_twitter_tweet_count"
  ],
  "Ethereum": [
    "open",
    "high",
    "low",
    "close",
    "volume",
    "pct_change_raw"
  ],
  "Dogecoin": [
    "open",
    "high",
    "low",
    "close",
    "volume",
    "pct_change_raw"
  ]
}


## Cell 18 — Save cleaned outputs for the rest of the project

In [18]:
price.to_csv("price_clean.csv", index=False)
btc_twitter_daily.to_csv("bitcoin_twitter_daily.csv", index=False)
bitcoin_model_data.to_csv("bitcoin_model_data.csv", index=False)
ethereum_model_data.to_csv("ethereum_model_data.csv", index=False)
dogecoin_model_data.to_csv("dogecoin_model_data.csv", index=False)

with open("feature_sets_by_coin.json", "w", encoding="utf-8") as f:
    json.dump(feature_sets, f, indent=2)

print("✅ Saved:")
print(" - price_clean.csv")
print(" - bitcoin_twitter_daily.csv")
print(" - bitcoin_model_data.csv")
print(" - ethereum_model_data.csv")
print(" - dogecoin_model_data.csv")
print(" - feature_sets_by_coin.json")


✅ Saved:
 - price_clean.csv
 - bitcoin_twitter_daily.csv
 - bitcoin_model_data.csv
 - ethereum_model_data.csv
 - dogecoin_model_data.csv
 - feature_sets_by_coin.json
